In [ ]:
"""
Statistical comparison of four policies (MAPPO, SA-T, SA-NT, OUT-50%)
while respecting the paired/block design of the experiment
(50 common evaluation seeds shared across all policies).

Method:
  1. For each RL architecture (5 independently trained models from 5 training seeds),
     the cost for each evaluation seed is averaged across the 5 models to remove
     inter-model noise while preserving the shared evaluation-seed pairing.
     -> Result: one value for each (policy, evaluation seed) = 50×4 matrix.
  2. Friedman test (non-parametric repeated-measures test,
     block = evaluation seed) for the global comparison of the 4 policies.
  3. Planned pairwise Wilcoxon signed-rank tests (n=50) for the three
     predefined comparisons, with Holm-Bonferroni correction for
     multiple comparisons.
  4. As an additional robustness check: model-level paired tests
     (paired t-test and Wilcoxon signed-rank test, n=5,
     unit = training seed) on the same three comparisons.

Requirements:
  pip install pandas openpyxl scipy statsmodels
"""

import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

# ---------------------------------------------------------------------------
# 0) Configuration: file paths and system cost column name
# ---------------------------------------------------------------------------
RL_FILES = {
    "MAPPO": "All Result - MAPPO.xlsx",
    "SA-T":  "All Result-SA-T.xlsx",
    "SA-NT": "All Result-SA-NT.xlsx",
}
OUT_FILE = "All Result-OUT.xlsx"
OUT_POLICY = "S50"          # Best order-up-to level according to the paper
COST_COL = "SC_Network_Total_Cost"
SHEET = "Raw_Episode_Data"


def rank_biserial_wilcoxon(after, before):
    """Rank-biserial effect size for paired Wilcoxon signed-rank tests."""
    d = after - before
    d = d[d != 0]
    ranks = stats.rankdata(np.abs(d))
    r_plus = ranks[d > 0].sum()
    r_minus = ranks[d < 0].sum()
    return (r_plus - r_minus) / (r_plus + r_minus)


def load_raw(path):
    return pd.read_excel(path, sheet_name=SHEET)


# ---------------------------------------------------------------------------
# 1) Load data and create two aggregation levels
# ---------------------------------------------------------------------------
model_level = {}   # Average across 50 evaluation seeds for each training seed -> n=5
eval_level = {}    # Average across 5 models for each evaluation seed -> n=50

for name, fname in RL_FILES.items():
    df = load_raw(fname)
    model_level[name] = df.groupby("Train_Seed")[COST_COL].mean().sort_index()
    eval_level[name] = df.groupby("Eval_Seed")[COST_COL].mean().sort_index()

df_out = load_raw(OUT_FILE)
out50 = (df_out[df_out["Train_Seed"] == OUT_POLICY]
         .set_index("Eval_Seed")[COST_COL].sort_index())
eval_level["OUT-50"] = out50   # OUT has no training seed; directly use n=50

eval_matrix = pd.DataFrame(eval_level)   # Shape: 50 x 4

print("=" * 70)
print("Overall mean cost of each policy (for consistency check with paper tables):")
print(eval_matrix.mean().round(2))

# ---------------------------------------------------------------------------
# 2) Global Friedman test (block = evaluation seed, n=50)
# ---------------------------------------------------------------------------
stat, p_friedman = stats.friedmanchisquare(
    eval_matrix["MAPPO"], eval_matrix["SA-T"],
    eval_matrix["SA-NT"], eval_matrix["OUT-50"]
)
avg_ranks = eval_matrix.rank(axis=1, method="average").mean().sort_values()

print("\n" + "=" * 70)
print("Global Friedman test (4 policies × 50 evaluation seeds):")
print(f"  chi2 = {stat:.3f}   p = {p_friedman:.3e}")
print("\nAverage ranks (lower = lower cost = better):")
print(avg_ranks.round(3))

# ---------------------------------------------------------------------------
# 3) Planned pairwise post-hoc tests (n=50) + Holm correction
# ---------------------------------------------------------------------------
comparisons = [("MAPPO", "SA-T"), ("SA-T", "SA-NT"), ("SA-NT", "OUT-50")]

rows = []
for a, b in comparisons:
    x, y = eval_matrix[a], eval_matrix[b]
    diff = y - x
    w_stat, p_w = stats.wilcoxon(x, y)
    r_rb = rank_biserial_wilcoxon(y, x)
    n_favor_a = int((diff > 0).sum())   # diff = b - a > 0 => a has lower cost => a wins
    rows.append({
        "comparison": f"{b} vs {a}",
        "mean_diff": round(diff.mean(), 2),
        "winner": a,
        "n_favor_winner_of_50": f"{n_favor_a}/50",
        "wilcoxon_W": w_stat,
        "wilcoxon_p": p_w,
        "effect_size_r": round(r_rb, 3),
    })

posthoc = pd.DataFrame(rows)
_, p_holm, _, _ = multipletests(posthoc["wilcoxon_p"], method="holm")
posthoc["p_holm"] = p_holm
posthoc["significant_at_0.05"] = posthoc["p_holm"] < 0.05

print("\n" + "=" * 70)
print("Pairwise Wilcoxon signed-rank post-hoc tests (n=50, block=evaluation seed):")
pd.set_option("display.width", 160)
print(posthoc.to_string(index=False))

# ---------------------------------------------------------------------------
# 4) Additional robustness check: model-level analysis
#    (n=5, unit = training seed)
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("Robustness check - model level, n=5 (unit = training seed):")
model_matrix = pd.DataFrame(model_level)

for a, b in [("MAPPO", "SA-T"), ("SA-T", "SA-NT")]:
    x, y = model_matrix[a], model_matrix[b]
    diff = y - x   # diff > 0 means b has higher cost, therefore a wins
    t_stat, p_t = stats.ttest_rel(y, x)

    try:
        w_stat, p_w = stats.wilcoxon(x, y, mode="exact")
    except ValueError:
        w_stat, p_w = np.nan, np.nan

    n_favor_a = int((diff > 0).sum())

    print(f"\n  {b} vs {a}:")
    print(f"    Mean difference = {diff.mean():.2f}   "
          f"({n_favor_a}/5 seeds in favor of {a})")
    print(f"    Paired t-test:      t={t_stat:.3f}, p={p_t:.4f}")
    print(f"    Wilcoxon (exact):   W={w_stat}, p={p_w:.4f}  "
          f"(Note: the minimum possible p-value with n=5 is 0.0625)")

print("\n" + "=" * 70)
print("Analysis completed.")